# 🔬 Kodomo Exercises: PDB Structure Analysis
# Упражнения Кодомо: Анализ PDB структур

---

Based on MSU Kodomo Bioinformatics Program (pr10)

## 🎯 Learning Objectives

- Understand PDB file format
- Extract atom coordinates
- Calculate center of mass
- Analyze protein chains
- Work with Cα atoms

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This assignment notebook is designed for hands-on practice of production-relevant analysis patterns.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Start with tiny inputs first, then scale after outputs look correct.
- Document assumptions for every threshold or filtering decision.
- Debug shape/type/path issues early to avoid cascading errors.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## 📊 PDB File Format

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         PDB ATOM RECORD FORMAT                               │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  Column    Content                                                          │
│  ──────────────────────────────────────────────────────────────────────     │
│  1-6       Record name (ATOM or HETATM)                                     │
│  7-11      Atom serial number                                               │
│  13-16     Atom name (CA, CB, N, C, O...)                                   │
│  17        Alternate location indicator                                     │
│  18-20     Residue name (ALA, GLY, MET...)                                  │
│  22        Chain identifier (A, B, C...)                                    │
│  23-26     Residue sequence number                                          │
│  31-38     X coordinate (Å)                                                 │
│  39-46     Y coordinate (Å)                                                 │
│  47-54     Z coordinate (Å)                                                 │
│  55-60     Occupancy                                                        │
│  61-66     Temperature factor (B-factor)                                    │
│  77-78     Element symbol                                                   │
│                                                                             │
│  Example:                                                                   │
│  ATOM      1  N   MET A   1      20.154  33.286   5.226  1.00 42.77      N  │
│  ATOM      2  CA  MET A   1      19.039  32.925   6.114  1.00 39.65      C  │
│       ↑    ↑   ↑   ↑   ↑   ↑      ↑       ↑       ↑      ↑    ↑           ↑ │
│       │    │   │   │   │   │      X       Y       Z      occ  B-factor   elem│
│       │    │   │   │   │   └── residue number                               │
│       │    │   │   │   └────── chain ID                                     │
│       │    │   │   └────────── residue name                                 │
│       │    │   └────────────── atom name                                    │
│       │    └────────────────── atom number                                  │
│       └─────────────────────── record type                                  │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## Exercise 1: PDB Parser - Extract Cα Atoms

**Task:** Extract Cα (alpha carbon) coordinates from a PDB file.

Based on `Kravchenko_pr10_pdb2.py` and `reader.py`

In [ ]:
def parse_pdb_ca_atoms(pdb_content):
    """
    Extract Cα atom information from PDB content.
    
    Based on Kodomo reader.py:
    - Check if line contains "ATOM  "
    - Check if atom name is " CA "
    - Extract: residue, chain, number, X, Y, Z
    
    Parameters:
        pdb_content: String content of PDB file
        
    Returns:
        List of dictionaries with Cα information
    """
    ca_atoms = []
    
    for line in pdb_content.split('\n'):
        # Check for ATOM record
        if line.startswith('ATOM  '):
            # Check for Cα atom (columns 13-16)
            atom_name = line[12:16]
            if atom_name == ' CA ':
                # Extract information using fixed-width columns
                residue = line[17:20].strip()     # Residue name
                chain = line[21]                   # Chain ID
                res_num = int(line[22:26].strip()) # Residue number
                x = float(line[30:38])             # X coordinate
                y = float(line[38:46])             # Y coordinate
                z = float(line[46:54])             # Z coordinate
                
                ca_atoms.append({
                    'residue': residue,
                    'chain': chain,
                    'res_num': res_num,
                    'x': x,
                    'y': y,
                    'z': z
                })
    
    return ca_atoms

# Sample PDB data
sample_pdb = """HEADER    HYDROLASE                               01-JAN-00   1CRN
ATOM      1  N   THR A   1      17.047  14.099   3.625  1.00 13.79           N
ATOM      2  CA  THR A   1      16.967  12.784   4.338  1.00 10.80           C
ATOM      3  C   THR A   1      15.685  12.755   5.133  1.00  9.19           C
ATOM      4  O   THR A   1      15.268  13.825   5.594  1.00  9.85           O
ATOM      5  N   THR A   2      15.115  11.545   5.265  1.00  7.81           N
ATOM      6  CA  THR A   2      13.856  11.469   6.066  1.00  8.31           C
ATOM      7  C   THR A   2      14.164  10.785   7.379  1.00  5.80           C
ATOM      8  N   CYS A   3      14.993   9.735   7.345  1.00  6.34           N
ATOM      9  CA  CYS A   3      15.385   8.961   8.529  1.00  5.39           C
"""

# Parse sample
ca_atoms = parse_pdb_ca_atoms(sample_pdb)
print("Extracted Cα atoms:")
print("-" * 50)
for atom in ca_atoms:
    print(f"{atom['residue']:3s} {atom['chain']} {atom['res_num']:4d}  "
          f"({atom['x']:8.3f}, {atom['y']:8.3f}, {atom['z']:8.3f})")

---

## Exercise 2: Calculate Center of Mass

**Task:** Calculate the geometric center (centroid) of all Cα atoms.

Based on `Kravchenko_pr10_centre.py`

```
┌─────────────────────────────────────────────────────────────────┐
│                   CENTER OF MASS FORMULA                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│         Σ xᵢ           Σ yᵢ           Σ zᵢ                      │
│  X̄ = ────────   Ȳ = ────────   Z̄ = ────────                     │
│           N              N              N                       │
│                                                                 │
│  Where N = number of atoms                                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
def calculate_centroid(ca_atoms):
    """
    Calculate geometric center of Cα atoms.
    
    Based on Kodomo pr10_centre:
    - Sum all X, Y, Z coordinates
    - Divide by number of atoms
    
    Parameters:
        ca_atoms: List of atom dictionaries with x, y, z keys
        
    Returns:
        Tuple (center_x, center_y, center_z)
    """
    if not ca_atoms:
        return (0.0, 0.0, 0.0)
    
    sum_x = sum(atom['x'] for atom in ca_atoms)
    sum_y = sum(atom['y'] for atom in ca_atoms)
    sum_z = sum(atom['z'] for atom in ca_atoms)
    
    n = len(ca_atoms)
    
    return (sum_x / n, sum_y / n, sum_z / n)

# Calculate center of our sample
center = calculate_centroid(ca_atoms)
print(f"Centroid of {len(ca_atoms)} Cα atoms:")
print(f"  X: {center[0]:.3f} Å")
print(f"  Y: {center[1]:.3f} Å")
print(f"  Z: {center[2]:.3f} Å")

---

## Exercise 3: Chain Statistics

**Task:** Count residues and chains in a PDB file.

Based on `Kravchenko_pr10_table1.py`

In [ ]:
def analyze_chains(ca_atoms):
    """
    Analyze chains in protein structure.
    
    Parameters:
        ca_atoms: List of Cα atom dictionaries
        
    Returns:
        Dictionary with chain statistics
    """
    chains = {}
    
    for atom in ca_atoms:
        chain_id = atom['chain']
        if chain_id not in chains:
            chains[chain_id] = []
        chains[chain_id].append(atom['residue'])
    
    return {
        'total_residues': len(ca_atoms),
        'num_chains': len(chains),
        'chains': {ch: len(res) for ch, res in chains.items()}
    }

# Analyze sample
stats = analyze_chains(ca_atoms)
print(f"Total residues: {stats['total_residues']}")
print(f"Number of chains: {stats['num_chains']}")
for chain, count in stats['chains'].items():
    print(f"  Chain {chain}: {count} residues")

---

## Exercise 4: Per-Chain Centroid

**Task:** Calculate separate centroids for each chain.

Based on `Kravchenko_pr10_centres.py`

In [ ]:
def calculate_chain_centroids(ca_atoms):
    """
    Calculate centroid for each chain separately.
    
    Parameters:
        ca_atoms: List of Cα atom dictionaries
        
    Returns:
        Dictionary {chain_id: (x, y, z)}
    """
    # Group atoms by chain
    chains = {}
    for atom in ca_atoms:
        chain_id = atom['chain']
        if chain_id not in chains:
            chains[chain_id] = []
        chains[chain_id].append(atom)
    
    # Calculate centroid for each chain
    centroids = {}
    for chain_id, atoms in chains.items():
        centroids[chain_id] = calculate_centroid(atoms)
    
    return centroids

# Multi-chain example
multichain_pdb = """ATOM      1  CA  ALA A   1      10.000  10.000  10.000  1.00 20.00           C
ATOM      2  CA  GLY A   2      12.000  10.000  10.000  1.00 20.00           C
ATOM      3  CA  VAL A   3      14.000  10.000  10.000  1.00 20.00           C
ATOM      4  CA  MET B   1      20.000  20.000  20.000  1.00 20.00           C
ATOM      5  CA  LEU B   2      22.000  20.000  20.000  1.00 20.00           C
"""

multi_atoms = parse_pdb_ca_atoms(multichain_pdb)
chain_centers = calculate_chain_centroids(multi_atoms)

print("Per-chain centroids:")
for chain, center in chain_centers.items():
    print(f"  Chain {chain}: ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f})")

---

## Exercise 5: Distance Calculation

**Task:** Calculate distance between two atoms or two centroids.

In [ ]:
import math

def calculate_distance(point1, point2):
    """
    Calculate Euclidean distance between two 3D points.
    
    Parameters:
        point1, point2: Tuples (x, y, z)
        
    Returns:
        Distance in Ångströms
    """
    dx = point1[0] - point2[0]
    dy = point1[1] - point2[1]
    dz = point1[2] - point2[2]
    
    return math.sqrt(dx**2 + dy**2 + dz**2)

# Calculate distance between chain centroids
if len(chain_centers) >= 2:
    chains = list(chain_centers.keys())
    dist = calculate_distance(chain_centers[chains[0]], chain_centers[chains[1]])
    print(f"Distance between Chain {chains[0]} and Chain {chains[1]} centroids: {dist:.2f} Å")

---

## 🏋️ Practice Exercises

### Exercise A: B-factor Analysis
Extract and analyze B-factors (temperature factors) from PDB file.

### Exercise B: Residue Counter
Count occurrences of each amino acid type in the structure.

### Exercise C: Contact Map
Generate a contact map showing which residues are within 8Å of each other.

---

## 📚 Key Takeaways

1. **PDB format**: Fixed-width columns, not delimited
2. **Cα atoms**: Alpha carbons represent backbone position
3. **Chain analysis**: Group atoms by chain ID
4. **Centroid**: Average of all coordinates